In [ ]:
# Replace the following with your path
%cd "/path/to/your/folder"

In [ ]:
import json
import time


In [ ]:
def process_nctid(nctid):

    out_link=f"out_last_time/{nctid}.json"

    
    # Remove duplicates for the same outcome
    # Load the JSON data
    with open(out_link, 'r') as file:
        data = json.load(file)

    # Remove duplicates while preserving order
    seen = set()
    unique_data = []
    for item in data:
        key = (item['Outcome measure'], item['Outcome description'], item['Time frame'], item['Classes'])  # Key based on unique fields
        if key not in seen:
            seen.add(key)
            unique_data.append(item)


    # Function to check if outcomes have the same measure, description, and time frame
    def same_outcome(outcomes):
        return len(set((outcome['Identifier']) for outcome in outcomes)) == 1

    # Function to merge the last two fields
    def merge_fields(outcomes):
        merged = outcomes[0].copy()

            # Collect unique values for the last two fields
        outcome_analysis_types = set()
        statistical_methods = set()

        for outcome in outcomes:
            outcome_analysis_types.add(outcome.get('Outcome Analysis Parameter Type', ''))
            statistical_methods.add(outcome.get('Statistical Analysis Method', ''))

            # Update the merged outcome with the collected values
        merged['Outcome Analysis Parameter Type'] = "; ".join(outcome_analysis_types)
        merged['Statistical Analysis Method'] = "; ".join(statistical_methods)

        return merged

    outcome_groups = []
    current_group = [unique_data[0]]

    for outcome in unique_data[1:]:
        if same_outcome(current_group + [outcome]):
            current_group.append(outcome)
        else:
            outcome_groups.append(current_group)
            current_group = [outcome]

    # Make sure the last group is added to the outcome_groups list
    if current_group:
        outcome_groups.append(current_group)


    outcome_groups

    # Merge each group and create the new data
    new_data = []
    for group in outcome_groups:
        if len(group) > 1:
            new_data.append(merge_fields(group))
        else:
            new_data.extend(group)


    # Save the updated data back to a JSON file
    with open(f'out_last_time/{nctid}_merged.json', 'w') as file:
        json.dump(new_data, file, indent=4)

In [ ]:
# Read NCTIDs from a text file
with open('nctid_new150.txt', 'r') as file:
    nctids = [line.strip() for line in file if line.strip()]

# Process each NCTID
# Initialize a counter variable
n = 1

for nctid in nctids:
    print(f"Processing NCTID {n}/{len(nctids)}: {nctid}")
    process_nctid(nctid)
    print(f"Finished processing NCTID: {nctid}")
    
    # Optional: Add a delay between processing each NCTID to avoid rate limiting
    time.sleep(5)
    
    # Increment the counter
    n += 1

print("All NCTIDs processed.")